# Sticky rod: pressure, time, and pull-off

A research note for an elastic rod against a rigid wall. **Compression creates active bonds; surviving-bond traction switches their loss on.** Nominal traction is the measured pulling-stress output. The spring is linear at fixed bond fraction, and beta is the only evolving state. This is an illustrative, isothermal, quasi-static model, not a copper calibration.

Read [the model and implementation PDF](model.pdf) for the connected derivation, assumptions, and rationale. Its [LaTeX source](model.tex) is kept alongside this note.

**Use this note:** run all cells once to populate the reference examples and controls. The playground below runs only when you press a button. All plots, tables, and animation remain here; nothing is exported. Save the notebook to retain displayed results. Saved animation needs notebook trust; recomputation needs a live kernel.

In [ ]:
%matplotlib inline
from dataclasses import asdict, replace
from pathlib import Path
from html import escape
import hashlib
import importlib.metadata
import json
import sys
from subprocess import SubprocessError

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import imageio_ffmpeg
from IPython.display import HTML, Markdown, clear_output, display
import sticky_rod
from sticky_rod import (Parameters, animate, plot_history, plot_sweep,
                        simulate, summarize_cycles, sweep)

print('Interpreter:', sys.executable)
print('Python:', sys.version.split()[0])
for package in ('numpy', 'matplotlib', 'ipywidgets', 'ipykernel', 'imageio-ffmpeg'):
    print(f'{package}: {importlib.metadata.version(package)}')
print('Core SHA256:', hashlib.sha256(Path(sticky_rod.__file__).read_bytes()).hexdigest())
print('Deterministic model; no random seed or stochastic sampling.')

## Visible Stickiness preset

The note starts with **one physical cycle**, using a compliant rod and a much stiffer adhesive spring. The player's loop button can replay it. Increase `cycles` only to simulate repeated loading on the same interface, carrying beta through return and recontact instead of resetting it. This visual demonstration overrides the Python model's unchanged defaults. All values use consistent arbitrary force, length, and time units.

Here `E=5` and `K_n=250` give a fully bonded interface/rod stiffness ratio of **50**. Initially, about 98% of the actuator's pulling motion stretches the rod instead of opening the gap. As beta decays, the tip retreats and the rod shortens. The animation magnifies displacements **10x**, while numeric strain, stress, and gap labels stay physical. This is elastic relaxation, not an inertial snap.

The bonded-area onset `bond_sigma0=0.02` protects a short elastic interval. During an activated open hold, surviving-bond traction increases as the rod relaxes, so rupture continues even while nominal traction falls. The eight-time-unit open hold lets beta and rod stretch approach zero without a latch or cutoff. The thresholds now describe bonded-area stress; they are not nominal tensile-strength values.

`p_peak` and `p_ref` are nominal pressures; `bond_sigma0` and `bond_delta_sigma` are bonded-area stresses. `K_n` is stress per length, not total spring stiffness. `v_pull` is length per time, and the rates are inverse time. The pressure ramp and compression unloading are separate from tensile pulling, so varying peak pressure does not change `v_pull`.

```text
nominal_traction = beta * K_n * gap
bond_traction    =        K_n * gap
formation_rate  = k_f * pressure / p_ref
rupture_rate    = k_d * smooth_switch(bond_traction)
d_beta_dt       = formation_rate * (1 - beta) - rupture_rate * beta
```

This assumes uniform parallel bonds. For the linear spring it is a stress-scaled gap law, not new microscopic physics. `bond_traction` is evaluated directly, never by dividing by a small beta. At beta=0 it is a limiting trial demand, not an actual stress on nonexistent bonds; force and bond loss remain zero.

In [ ]:
base = Parameters(
    E=5.0, A=1.0, L=1.0, K_n=250.0,
    k_f=1.0, p_ref=0.05, k_d=2.0, bond_sigma0=0.02, bond_delta_sigma=0.01,
    beta0=0.0, p_peak=0.05,
    t_approach=1.0, t_press=1.0, t_hold=4.0, t_unload=1.0,
    pull_distance=0.08, v_pull=0.015, t_open_hold=8.0, cycles=1,
    dt=0.01, rtol=1e-5, atol=1e-9,
)

def show_inputs(p, title='Exact inputs for this result'):
    body = escape(json.dumps(asdict(p), indent=2))
    display(HTML(f'<details><summary>{escape(title)}</summary><pre>{body}</pre></details>'))

def show_rows(rows, columns):
    def text(value):
        return format(value, '.6g') if isinstance(value, (float, np.floating)) else str(value)
    header = ''.join(f'<th>{escape(key)}</th>' for key in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(text(row[key]))}</td>' for key in columns)
                   + '</tr>' for row in rows)
    display(HTML(f'<div style="overflow-x:auto"><table><thead><tr>{header}</tr></thead>'
                 f'<tbody>{body}</tbody></table></div>'))

def show_history(sol, p):
    show_inputs(p)
    rows = summarize_cycles(sol, p)
    show_rows(rows, ['cycle', 'beta_start', 'beta_pull_start', 'peak_stress',
                     'beta_end', 'end_stress', 'pressure_dose', 'peak_at_pull_end'])
    fig = plot_history(sol, p)
    display(fig)
    plt.close(fig)
    print('Maximum rod strain |N|/(EA):', float(np.max(np.abs(sol['N'])) / (p.E * p.A)))
    print('This strain must remain small for the elastic rod approximation.')
    print('Solver statistics:', sol['stats'])

def inline_animation(sol, p, frames=480, fps=20, displacement_scale=10.0):
    with plt.rc_context({'figure.dpi': 75, 'animation.embed_limit': 48,
                         'animation.writer': 'ffmpeg', 'animation.bitrate': 1200,
                         'animation.ffmpeg_args': ['-movflags', '+faststart', '-threads', '1'],
                         'animation.ffmpeg_path': imageio_ffmpeg.get_ffmpeg_exe()}):
        movie = animate(sol, p, frames=frames, fps=fps, displacement_scale=displacement_scale)
        try:
            html = movie.to_html5_video()
            if 'data:video/mp4;base64,' not in html:
                raise RuntimeError('Embedded animation is too large; reduce the frame count.')
            loop_control = '''<label><input type=checkbox onchange="this.closest('div').querySelector('video').loop=this.checked"> Loop this cycle</label>'''
            return HTML('<div>' + html.replace(' autoplay', '', 1) + loop_control + '</div>')
        except (OSError, SubprocessError) as exc:
            raise RuntimeError('Video encoding failed; check the uv environment and encoder.') from exc
        finally:
            plt.close(movie._fig)

show_inputs(base, 'Reference parameters (expand)')

## Approach, bond, pull, and return

Each cycle starts separated, approaches the wall, presses, holds under pressure, unloads compression, pulls at fixed speed, and holds the actuator open. The next approach is the return of the same interface. A positive gap does **not** mean complete debonding.

`peak_at_pull_end=True` flags a positive peak at the end of the prescribed pull, not a demonstrated pull-off strength. Increase the pulling range and refine time resolution before interpreting such a peak as failure.

In [ ]:
reference = simulate(base)
show_history(reference, base)

### Why rupture continues as measured force falls

During the fixed-actuator open hold, compare the falling nominal traction with the surviving-bond traction. They use separate panels because their scales can be very different. The constant activation thresholds belong only on the bonded-area panel; the equivalent nominal onset is `beta * bond_sigma0`. No historical switch is latched.

The right panel uses a logarithmic axis and omits zero traction and states with no bonds. The saved arrays still contain those states. Deliberate closure can unload surviving bonds and stop loss; renewed compression can form bonds again.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].plot(reference['t'], reference['traction'], label='Nominal traction')
axes[0].plot(reference['t'], reference['beta'] * base.bond_sigma0, ':',
             label='Equivalent nominal onset')
axes[0].set(title='Measured interface response', xlabel='Time', ylabel='Nominal stress')
bonded = (reference['beta'] > 0) & (reference['bond_traction'] > 0)
axes[1].plot(reference['t'], np.where(bonded, reference['bond_traction'], np.nan),
             label='Surviving-bond traction')
axes[1].axhline(base.bond_sigma0, linestyle=':', label='Bonded-area onset')
axes[1].axhline(base.bond_sigma0 + base.bond_delta_sigma, linestyle='--', label='Full activation')
axes[1].set(title='Rupture driver (log scale)', xlabel='Time', ylabel='Bonded-area stress', yscale='log')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize='small')
display(fig)
plt.close(fig)

### Animation of the computed cycles

Press Play in the embedded player. Displacements are magnified **10x** to make the sticky effect visible. The rod extends beyond the dashed, unstretched-length guide during tension, then shortens as bonds decay. Its color shows **signed axial stress N/A**: blue is compression, the neutral middle is zero stress, and red is tension. The labelled colorbar uses one fixed symmetric range for the whole trajectory, so relaxation fades toward neutral rather than being rescaled each frame. The stress is uniform along this one-element rod; there is no artificial spatial gradient.

The dashed guide is anchored at the actuator and is not a second physical rod; it may cross the wall during compression. Orange adhesive opacity follows beta, separately from rod stress. The red arrow indicates force direction at the right tip, **not magnitude**. The legend identifies the guide and adhesive link. If a run has exactly zero stress throughout, the colorbar uses +/- p_ref as a reference span while the rod stays neutral.

All numeric labels and history curves show **actual** strain, gap, and stress. The scene reports both traction measures and when bond loss becomes active. The bonded-area onset is not a hard nominal-strength cap. This activated hold relaxes toward zero bonds and force; a subthreshold experiment need not debond. Playback uses **480 frames at 20 fps**, about **24 seconds per cycle**. The higher frame count makes motion smoother without speeding up the cycle or changing the solver time step. Physical time is shown separately. Check **Loop this cycle** below the video to replay; replay does not simulate additional cycles or carry state into the next replay.

The MP4 is embedded in this notebook with browser playback and seeking controls. Video compression keeps the larger frame count manageable without changing numerical results. The encoder comes from `imageio-ffmpeg` in the uv environment; no separate system installation or exported movie file is needed.

In [ ]:
display(inline_animation(reference, base, frames=480, fps=20))

## Interactive playground

Change the controls, then press **Run experiment**. Computation is not triggered by typing or moving a slider. The new history and its exact input snapshot appear below the controls; the reference examples above remain unchanged.

**Animate last run** uses the last successful experiment, not unrun control edits. Rerunning an experiment clears its previous interactive animation. The player itself never reruns the solver. Advanced inputs expose geometry, kinetics, all durations, and numerical resolution.

In [ ]:
units = dict(E='stress', A='area', L='length', K_n='stress/length',
             k_f='1/time', p_ref='stress', k_d='1/time', bond_sigma0='bond stress',
             bond_delta_sigma='bond stress', p_peak='stress', t_approach='time',
             t_press='time', t_hold='time', t_unload='time',
             pull_distance='length', v_pull='length/time', t_open_hold='time', dt='time')
controls = {}
for key, value in asdict(base).items():
    description = key + (f' [{units[key]}]' if key in units else '')
    options = dict(description=description, style={'description_width': 'initial'},
                   layout=widgets.Layout(width='340px'))
    controls[key] = (widgets.BoundedIntText(value=value, min=1, max=24, **options)
                     if key == 'cycles' else widgets.FloatText(value=value, **options))
primary = ('p_peak', 't_hold', 'v_pull', 'bond_sigma0', 'bond_delta_sigma', 'k_d', 'cycles')
primary_box = widgets.Box([controls[key] for key in primary],
                          layout=widgets.Layout(display='flex', flex_flow='row wrap'))
advanced = widgets.Accordion(children=[widgets.Box(
    [control for key, control in controls.items() if key not in primary],
    layout=widgets.Layout(display='flex', flex_flow='row wrap'))], selected_index=None)
advanced.set_title(0, 'Geometry, formation, loading, and numerical resolution')
run_button = widgets.Button(description='Run experiment', button_style='primary')
animation_button = widgets.Button(description='Animate last run')
frame_control = widgets.IntSlider(value=480, min=24, max=600, step=24, description='Frames')
fps_control = widgets.IntSlider(value=20, min=1, max=30, description='Playback fps')
scale_control = widgets.IntSlider(value=10, min=1, max=20, description='Displacement scale',
                                  style={'description_width': 'initial'})
experiment_output = widgets.Output()
animation_output = widgets.Output()
current = {'p': base, 'sol': reference}

def selected_parameters():
    return Parameters(**{key: control.value for key, control in controls.items()})

def run_experiment(_=None):
    run_button.disabled = True
    try:
        p = selected_parameters()
        sol = simulate(p)
        with experiment_output:
            clear_output(wait=True)
            show_history(sol, p)
        current.update(p=p, sol=sol)
        with animation_output:
            clear_output()
            print('New trajectory ready. Press Animate last run to view it.')
    except (ValueError, RuntimeError, OverflowError) as exc:
        with experiment_output:
            clear_output(wait=True)
            print('Experiment failed; last successful trajectory retained:', exc)
    finally:
        run_button.disabled = False

def run_animation(_=None):
    animation_button.disabled = True
    try:
        html = inline_animation(current['sol'], current['p'], frame_control.value, fps_control.value,
                                displacement_scale=scale_control.value)
        with animation_output:
            clear_output(wait=True)
            show_inputs(current['p'], 'Inputs of the animated last successful run')
            display(html)
    except (ValueError, RuntimeError, OverflowError) as exc:
        with animation_output:
            clear_output(wait=True)
            print('Animation failed:', exc)
    finally:
        animation_button.disabled = False

run_button.on_click(run_experiment)
animation_button.on_click(run_animation)
display(primary_box, advanced, run_button, experiment_output,
        widgets.Box([frame_control, fps_control, scale_control],
                    layout=widgets.Layout(display='flex', flex_flow='row wrap')),
        animation_button, animation_output)
print('No new computation yet. The last successful run is the reference example above.')

## Pressure and hold-time studies

The reference studies below use **one cycle per independent case**, holding all inputs except the selected study variable fixed. Every case starts from the same beta0; it does not inherit the preceding case's bonds.

The current formation law combines pressure and time through integrated exposure, including loading and unloading ramps: $Q_p=p_{\mathrm{peak}}(t_{\mathrm{press}}/2+t_{\mathrm{hold}}+t_{\mathrm{unload}}/2)$. Equal exposure gives equal post-compression beta for the same initial compression state. These are not independent microscopic mechanisms in this minimal law.

Compare trends without confusing peak stress with complete separation. Squares mark loading-endpoint peaks. Runs that never activate rupture can retain bonds; an activated fixed-actuator hold continues losing them as nominal stress falls.

In [ ]:
study_base = replace(base, cycles=1)
pressure_values = np.linspace(0.0, 0.1, 7)
hold_values = np.linspace(0.0, 8.0, 7)
show_inputs(study_base, 'Fixed reference-study inputs (the swept field is replaced)')
reference_sweeps = {}
for field, values in [('p_peak', pressure_values), ('t_hold', hold_values)]:
    display(Markdown('### ' + ('Pressure at fixed hold time' if field == 'p_peak'
                              else 'Hold time at fixed peak pressure')))
    rows = sweep(study_base, field, values)
    reference_sweeps[field] = rows
    show_rows(rows, [field, 'cycle', 'beta_pull_start', 'peak_stress',
                     'beta_end', 'pressure_dose', 'peak_at_pull_end'])
    fig = plot_sweep(rows, field)
    display(fig)
    plt.close(fig)

### Configure another study

This button reads the **current playground controls**, including cycle count, and replaces only the selected sweep variable. It does not require a prior Run experiment. Each case remains independent; within a case, beta carries continuously through its cycles. The resulting plot has one curve per cycle. The saved input snapshot and table identify exactly what was computed.

In [ ]:
sweep_field = widgets.Dropdown(options=[('Peak pressure', 'p_peak'), ('Hold time', 't_hold')],
                               description='Vary')
sweep_min = widgets.FloatText(value=0.0, description='From')
sweep_max = widgets.FloatText(value=0.1, description='To')
sweep_count = widgets.BoundedIntText(value=7, min=2, max=32, description='Cases')
sweep_button = widgets.Button(description='Run study', button_style='primary')
sweep_output = widgets.Output()
latest_study = {}

def change_study_range(change):
    sweep_min.value = 0.0
    sweep_max.value = 0.1 if change['new'] == 'p_peak' else 8.0

def run_study(_=None):
    sweep_button.disabled = True
    try:
        p = selected_parameters()
        field = sweep_field.value
        if not np.isfinite([sweep_min.value, sweep_max.value]).all() or sweep_min.value >= sweep_max.value:
            raise ValueError('Study bounds must be finite, with From less than To.')
        values = np.linspace(sweep_min.value, sweep_max.value, sweep_count.value)
        rows = sweep(p, field, values)
        with sweep_output:
            clear_output(wait=True)
            show_inputs(p, f'Study inputs; {field} is replaced by each table value')
            show_rows(rows, [field, 'cycle', 'beta_pull_start', 'peak_stress',
                             'beta_end', 'pressure_dose', 'peak_at_pull_end'])
            fig = plot_sweep(rows, field)
            display(fig)
            plt.close(fig)
        latest_study.update(p=p, field=field, values=values, rows=rows)
    except (ValueError, RuntimeError, OverflowError) as exc:
        with sweep_output:
            clear_output(wait=True)
            print('Study failed:', exc)
    finally:
        sweep_button.disabled = False

sweep_field.observe(change_study_range, names='value')
sweep_button.on_click(run_study)
display(widgets.Box([sweep_field, sweep_min, sweep_max, sweep_count],
                    layout=widgets.Layout(display='flex', flex_flow='row wrap')),
        sweep_button, sweep_output)

## Numerical checks

Equilibrium is exact up to roundoff but beta is time-integrated. The checks below compare the reference cycles to a finer run, verify the analytic compression-stage pressure exposure, and report force balance. They test implementation, not copper physics. Local integration tolerances alone do not guarantee peak accuracy; use a still finer study when drawing conclusions.

In [ ]:
fine_p = replace(base, dt=base.dt / 2, rtol=base.rtol / 4, atol=base.atol / 4)
fine = simulate(fine_p)
beta_difference = float(np.max(np.abs(reference['beta'] - np.interp(reference['t'], fine['t'], fine['beta']))))
rows = summarize_cycles(reference, base)
fine_rows = summarize_cycles(fine, fine_p)
checks = []
for row, refined in zip(rows, fine_rows):
    approach = next(s for s in reference['segments']
                    if s['cycle'] == row['cycle'] and s['phase'] == 'approach')
    beta_after_approach = float(np.interp(approach['end'], reference['t'], reference['beta']))
    exact = 1 - (1 - beta_after_approach) * np.exp(-base.k_f * row['pressure_dose'] / base.p_ref)
    checks.append(dict(cycle=row['cycle'], peak=row['peak_stress'], fine_peak=refined['peak_stress'],
                       peak_difference=abs(row['peak_stress'] - refined['peak_stress']),
                       formation_error=abs(row['beta_pull_start'] - exact),
                       balance_error=row['max_force_balance_error']))
show_inputs(fine_p, 'Refined-run inputs')
show_rows(checks, ['cycle', 'peak', 'fine_peak', 'peak_difference', 'formation_error', 'balance_error'])
print('Maximum beta-history difference on reference output times:', beta_difference)
print('Maximum contact complementarity error:', max(row['max_complementarity_error'] for row in rows))
assert np.all((reference['beta'] >= 0) & (reference['beta'] <= 1))
assert beta_difference < 0.003, 'Refine the reference integration.'
assert all(row['formation_error'] < 0.001 for row in checks), 'Compression integration needs refinement.'
assert max(row['max_complementarity_error'] for row in rows) == 0
print('Reference bounds, pressure-exposure, and time-refinement checks passed.')

### Regression suite

The independent tests also check switch and residual derivatives, the opening-step uniqueness safeguard, continued activated loss despite falling nominal stress, closure and rebonding, tiny beta, area scaling, cycle continuity, sweep independence, invalid inputs, and animation geometry. Rerun this cell after changing the Python model (restart the kernel first to reload it).

In [ ]:
import io
import unittest
suite = unittest.defaultTestLoader.discover(str(Path(sticky_rod.__file__).parent),
                                           pattern='test_sticky_rod.py')
test_log = io.StringIO()
test_result = unittest.TextTestRunner(stream=test_log, verbosity=1).run(suite)
print(test_log.getvalue())
assert test_result.wasSuccessful(), 'Regression tests failed; inspect the output above.'